# nanoVLM JAX Benchmark: MMStar Math Sanity Check

This notebook compares the JAX implementation of nanoVLM (loaded via `from_pretrained`) against the reported PyTorch baseline on the **MMStar math** subset.


**Reference score from model card**:
```
mmstar_math: 0.390821304464047
```

In [1]:
# @title Configuration
MODEL_ID = "lusxvr/nanoVLM-230M-8k"  # @param {type:"string"}
BATCH_SIZE = 1  # @param {type:"integer"}
MAX_NEW_TOKENS = 5  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
REPORTED_MMSTAR_MATH = 0.390821304464047  # @param {type:"number"}

## 📦 Setup & Dependencies

In [2]:
%%capture
!pip install --upgrade jax[tpu] jaxlib -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install transformers datasets pillow safetensors numpy einops jaxtyping

In [ ]:
import json
import math
import os
import string
import sys
from collections import defaultdict
from typing import List, Tuple

import numpy as np
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoTokenizer

# Clone nanoVLMJAX repo
if not os.path.exists("nanoVLMJAX"):
    !git clone https://github.com/SauravMaheshkar/nanoVLMJAX.git

sys.path.insert(0, "nanoVLMJAX")

import jax
import jax.numpy as jnp

print("JAX devices:", jax.devices())

## 💿 Load MMStar Dataset

In [4]:
# Load MMStar dataset
ds = load_dataset("Lin-Chen/MMStar", split="val", trust_remote_code=True)

# Filter to math category
math_docs = [doc for doc in ds if doc["category"] == "math"]
print(f"Total MMStar samples: {len(ds)}")
print(f"MMStar math samples: {len(math_docs)}")

# Show an example
if math_docs:
    ex = math_docs[0]
    print("\nExample question:", ex["question"])
    print("Answer:", ex["answer"])
    print("L2 category:", ex["l2_category"])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Lin-Chen/MMStar' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Lin-Chen/MMStar' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

mmstar.parquet:   0%|          | 0.00/41.8M [00:00<?, ?B/s]

Generating val split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Total MMStar samples: 1500
MMStar math samples: 250

Example question: Hint: Please answer the question and provide the correct option letter, e.g., A, B, C, D, at the end.
Question: A square is tangent to a line at point P in the figure above. What is the value of x?
Choices:
(A) 30
(B) 15
(C) 20
(D) 45
Answer: A
L2 category: geometry


## 🖼️ Image Processing Utilities

In [13]:
class DynamicResizePIL:
    """Resize so longest side ≤ max_side_len and divisible by patch_size."""

    def __init__(
        self, patch_size: int, max_side_len: int, resize_to_max_side_len: bool = False
    ):
        self.p = patch_size
        self.m = max_side_len
        self.resize_to_max_side_len = resize_to_max_side_len

    def _get_new_hw(self, h: int, w: int) -> Tuple[int, int]:
        long, short = (w, h) if w >= h else (h, w)
        target_long = (
            self.m
            if self.resize_to_max_side_len
            else min(self.m, math.ceil(long / self.p) * self.p)
        )
        scale = target_long / long
        target_short = math.ceil(short * scale / self.p) * self.p
        target_short = max(target_short, self.p)
        return (target_short, target_long) if w >= h else (target_long, target_short)

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        new_h, new_w = self._get_new_hw(h, w)
        return img.resize((new_w, new_h), Image.BICUBIC)


class GlobalAndSplitImagesPIL:
    """Create global patch + split patches. Returns list of PIL images and grid."""

    def __init__(self, patch_size: int):
        self.p = patch_size

    def __call__(self, img: Image.Image) -> Tuple[List[Image.Image], Tuple[int, int]]:
        w, h = img.size
        assert h % self.p == 0 and w % self.p == 0, (
            f"Image size {(h, w)} not divisible by patch_size {self.p}"
        )
        n_h, n_w = h // self.p, w // self.p
        if n_h == 1 and n_w == 1:
            return [img], (n_h, n_w)
        global_patch = img.resize((self.p, self.p), Image.BICUBIC)
        patches = []
        for i in range(n_h):
            for j in range(n_w):
                patch = img.crop(
                    (j * self.p, i * self.p, (j + 1) * self.p, (i + 1) * self.p)
                )
                patches.append(patch)
        return [global_patch] + patches, (n_h, n_w)


def process_image_for_model(
    image: Image.Image,
    patch_size: int = 512,
    max_side_len: int = 2048,
    resize_to_max_side_len: bool = True,
    image_processor=None,
) -> Tuple[np.ndarray, Tuple[int, int]]:
    """Process a PIL image into patches ready for the vision encoder.

    Returns:
        pixel_values: (num_patches, C, patch_size, patch_size) numpy array
        grid: (n_h, n_w) number of split patches (excluding global)
    """
    if image.mode != "RGB":
        image = image.convert("RGB")
    resizer = DynamicResizePIL(patch_size, max_side_len, resize_to_max_side_len)
    resized = resizer(image)
    splitter = GlobalAndSplitImagesPIL(patch_size)
    patches, grid = splitter(resized)
    if image_processor is not None:
        inputs = image_processor(images=patches)
        pixel_values = inputs["pixel_values"]
        if hasattr(pixel_values, "numpy"):
            pixel_values = pixel_values.numpy()
    else:
        pixel_values = np.stack(
            [np.array(p).transpose(2, 0, 1) / 255.0 for p in patches], axis=0
        )
        pixel_values = (
            pixel_values - np.array([0.5, 0.5, 0.5]).reshape(1, 3, 1, 1)
        ) / np.array([0.5, 0.5, 0.5]).reshape(1, 3, 1, 1)
    return pixel_values, grid

## 📝 Tokenizer Setup

In [14]:
# Load model config to get tokenizer info and extra tokens
from huggingface_hub import hf_hub_download

config_path = hf_hub_download(repo_id=MODEL_ID, filename="config.json")
with open(config_path) as f:
    raw_cfg = json.load(f)

tokenizer_name = raw_cfg.get("lm_tokenizer", "HuggingFaceTB/SmolLM2-360M-Instruct")
chat_template = raw_cfg.get("lm_chat_template", None)
extra_tokens = raw_cfg.get("vlm_extra_tokens", {})
mp_image_token_length = raw_cfg.get("mp_image_token_length", 64)
vit_img_size = raw_cfg.get("vit_img_size", 512)
max_img_size = raw_cfg.get("max_img_size", 2048)
resize_to_max_side_len = raw_cfg.get("resize_to_max_side_len", True)

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Add extra tokens and set attributes to match original wrapper behavior
if extra_tokens:
    tokens_to_add = list(extra_tokens.values())
    tokenizer.add_tokens(tokens_to_add)
    for k, v in extra_tokens.items():
        setattr(tokenizer, k, v)

# Add chat template if provided
if chat_template:
    tokenizer.chat_template = chat_template

image_token_id = tokenizer.convert_tokens_to_ids("<|image|>")
global_image_token_id = (
    tokenizer.convert_tokens_to_ids("<|global_image|>")
    if "<|global_image|>" in tokenizer.get_vocab()
    else None
)
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Image token id: {image_token_id}")
print(f"Global image token id: {global_image_token_id}")

Tokenizer vocab size: 49218
Image token id: 49152
Global image token id: 49153


## 🤖 Load JAX Model

In [16]:
from src.models.language_model import language_model_forward
from src.models.modality_projector import modality_projector_forward
from src.models.vision_language_model import VisionLanguageModel
from src.models.vit import vit_forward

print("Loading JAX model from pretrained...")
jax_params = VisionLanguageModel.from_pretrained(MODEL_ID)
print("Model loaded.")

# Update image_token_id to match tokenizer (config may not store it)
jax_params.image_token_id = int(image_token_id)
print(f"Set jax_params.image_token_id = {jax_params.image_token_id}")

# Move to TPU
jax_params = jax.device_put(jax_params, jax.devices()[0])
print("Parameters placed on", jax_params.decoder.token_embedding.device)

# Extract config values from raw config
mp_image_token_length = raw_cfg.get("mp_image_token_length", 64)
vit_img_size = raw_cfg.get("vit_img_size", 512)

# Load image processor from the vision backbone
vit_model_type = raw_cfg.get("vit_model_type", "google/siglip2-base-patch16-512")
image_processor = AutoImageProcessor.from_pretrained(
    vit_model_type, trust_remote_code=True
)
print(f"Image processor: {vit_model_type}")

Loading JAX model from pretrained...


Model loaded.
Set jax_params.image_token_id = 49152
Parameters placed on TPU_0(process=0,(0,0,0,0))
Image processor: google/siglip2-base-patch16-512


## ⚙️ Custom Forward with Multi-Patch Support

In [17]:
def get_image_string_jax(
    tokenizer, splitted_image_counts: List[Tuple[int, int]], mp_image_token_length: int
) -> str:
    """Build image token string matching original nanoVLM format."""
    image_string = ""
    for idx, (n_h, n_w) in enumerate(splitted_image_counts):
        if len(splitted_image_counts) > 1:
            image_string += f"<image: {idx}>"
        if hasattr(tokenizer, "global_image_token"):
            image_string += tokenizer.global_image_token
            image_string += tokenizer.image_token * mp_image_token_length
            if n_h == 1 and n_w == 1:
                continue
        for i in range(n_h):
            for j in range(n_w):
                attr_name = f"r{i + 1}c{j + 1}"
                if hasattr(tokenizer, attr_name):
                    image_string += getattr(tokenizer, attr_name)
                else:
                    # Fallback if row/col tokens not in tokenizer
                    image_string += f"<|r{i + 1}c{j + 1}|>"
                image_string += tokenizer.image_token * mp_image_token_length
    return image_string


def vlm_forward_multi_image(params, input_ids, images, key, attention_mask):
    """Forward pass supporting multiple image patches per sample.

    Replaces image-token positions in input_ids with the corresponding
    vision embeddings from all patches. Fully vectorized so it works for
    any batch size and is JIT-compatible.
    """
    token_embd = params.decoder.token_embedding[input_ids]

    if images is not None:
        image_embd = vit_forward(params.vision_encoder, images, key)
        image_embd = modality_projector_forward(params.modality_projector, image_embd)
        # image_embd: (num_total_patches, tokens_per_patch, D_lm)
        flat_image_embd = image_embd.reshape(-1, image_embd.shape[-1])

        image_mask = input_ids == params.image_token_id
        # Per-sample counts and offsets into the flat embedding array
        counts = jnp.sum(image_mask, axis=1)
        offsets = jnp.concatenate(
            [jnp.zeros(1, dtype=counts.dtype), jnp.cumsum(counts)[:-1]]
        )

        # Sequential index of each image token within its sample
        local_indices = jnp.cumsum(image_mask, axis=1) - 1
        local_indices = jnp.where(image_mask, local_indices, 0)
        global_indices = offsets[:, None] + local_indices

        # Gather embeddings and scatter only at image-token positions
        gathered = flat_image_embd[global_indices]
        token_embd = jnp.where(image_mask[:, :, None], gathered, token_embd)

    logits = language_model_forward(
        params.decoder, input_ids, key, attention_mask, token_embd=token_embd
    )
    return logits


# JIT-compile the forward pass for efficient repeated generation steps.
_vlm_forward_jit = jax.jit(vlm_forward_multi_image)


def greedy_generate_jax(
    params,
    input_ids,
    images,
    attention_mask,
    max_new_tokens=5,
):
    """Greedy generation for JAX VLM with multi-patch support.

    The forward pass is JIT-compiled; JAX will cache a compiled version for
    each unique input shape, so with small max_new_tokens the overhead is
    negligible.
    """
    batch_size = input_ids.shape[0]
    generated = []

    for _ in range(max_new_tokens):
        logits = _vlm_forward_jit(params, input_ids, images, None, attention_mask)
        next_token = jnp.argmax(logits[:, -1, :], axis=-1)
        generated.append(next_token)
        input_ids = jnp.concatenate([input_ids, next_token[:, None]], axis=1)
        if attention_mask is not None:
            attention_mask = jnp.concatenate(
                [attention_mask, jnp.ones((batch_size, 1), dtype=attention_mask.dtype)],
                axis=1,
            )

    return jnp.stack(generated, axis=1)

## 🧪 Evaluate on MMStar Math

In [18]:
def extract_mcq_answer(pred: str, choices=None) -> str:
    """Extract multiple-choice letter from prediction."""
    if choices is None:
        choices = ["A", "B", "C", "D"]
    pred = pred.strip().upper()
    # Direct match
    if pred in choices:
        return pred
    # Find first occurrence of a choice letter
    for c in choices:
        if c in pred:
            return c
    return ""


def exact_match(pred, gt):
    gt_letter = gt.strip().upper()
    pred_letter = extract_mcq_answer(pred)
    return 1.0 if pred_letter == gt_letter else 0.0


def build_prompt(
    doc,
    tokenizer,
    mp_image_token_length: int,
    post_prompt: str = "\nAnswer with the option's letter from the given choices directly",  # noqa: E501
) -> Tuple[str, np.ndarray, Tuple[int, int]]:
    """Build prompt and image string for a single MMStar doc."""
    image = doc["image"].convert("RGB")
    pixel_values, grid = process_image_for_model(
        image,
        patch_size=vit_img_size,
        max_side_len=max_img_size,
        resize_to_max_side_len=resize_to_max_side_len,
        image_processor=image_processor,
    )

    # Build image string
    splitted_image_counts = [grid]
    image_string = get_image_string_jax(
        tokenizer, splitted_image_counts, mp_image_token_length
    )

    # Build question text
    question = doc["question"].strip()
    options = {cand: doc[cand] for cand in string.ascii_uppercase if cand in doc}
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])
    text = f"{question}\n{options_text}{post_prompt}"

    # Chat template
    messages = [{"role": "user", "content": image_string + text}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Add assistant prefix (matching original wrapper behavior)
    prompt = prompt + "Answer:"

    return prompt, pixel_values, grid


# Pre-compute all prompts and images
print("Preparing prompts and images...")
prepared = []
for doc in tqdm(math_docs):
    prompt, pixel_values, grid = build_prompt(doc, tokenizer, mp_image_token_length)
    prepared.append((doc, prompt, pixel_values, grid))
print(f"Prepared {len(prepared)} samples")

Preparing prompts and images...


100%|██████████| 250/250 [00:19<00:00, 12.63it/s]

Prepared 250 samples


In [ ]:
print("Running JAX inference on MMStar math...")
results = []

for doc, prompt, pixel_values, grid in tqdm(prepared, desc="JAX eval"):
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="np", truncation=True, max_length=4096)
    input_ids = jnp.array(inputs["input_ids"], dtype=jnp.int32)
    attention_mask = jnp.array(inputs["attention_mask"], dtype=jnp.int32)

    # Prepare images
    images_jax = jnp.array(pixel_values, dtype=jnp.float32)

    # Generate
    generated_ids = greedy_generate_jax(
        jax_params,
        input_ids,
        images_jax,
        attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # Decode
    pred_text = tokenizer.batch_decode(
        np.array(generated_ids), skip_special_tokens=True
    )[0]
    gt = doc["answer"]
    score = exact_match(pred_text, gt)
    results.append(
        {
            "question_id": doc["index"],
            "l2_category": doc["l2_category"],
            "pred": pred_text,
            "gt": gt,
            "score": score,
        }
    )

# Aggregate scores by l2_category
l2_scores = defaultdict(list)
for r in results:
    l2_scores[r["l2_category"]].append(r["score"])

l2_avg = {k: sum(v) / len(v) for k, v in l2_scores.items()}
overall = sum(l2_avg.values()) / len(l2_avg) if l2_avg else 0.0

print("\n=== JAX MMStar Math Results ===")
for k, v in l2_avg.items():
    print(f"  {k}: {v:.4f}")
print(f"\nOverall mmstar_math (JAX): {overall:.6f}")
print(f"Reported mmstar_math (PyTorch): {REPORTED_MMSTAR_MATH:.6f}")
print(f"Difference: {abs(overall - REPORTED_MMSTAR_MATH):.6f}")

Running JAX inference on MMStar math...


JAX eval:   2%|▏         | 5/250 [06:12<5:07:04, 75.20s/it]